In [1]:
!pip install crewai langchain openai!pip install pymupdf!pip install langchain-community openai --quiet!pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.5/119.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 

In [2]:
import osfrom crewai import LLMfrom langchain.memory import ConversationBufferMemoryos.environ["GROQ_API_KEY"] = "add_your_api_key"os.environ["HF_TOKEN"] = "add_your_api_key"os.environ['FIRECRAWL_API_KEY'] = 'fc-4f489a12b7ce47fda6737fb86e53ce27'llm = LLM(    model="groq/meta-llama/llama-4-maverick-17b-128e-instruct",    temperature=0.7)memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

/tmp/ipython-input-1529700501.py:14: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [3]:
from langchain.memory import ConversationBufferMemoryfrom crewai import Agent, Task, Crewmedi_pal = Agent(    role="Disease Information Counselor",    goal="Provide informative and empathetic responses about health topics",    backstory=(        "You're MediPal, a kind, friendly health info assistant. You’re not a doctor, "        "but you help users understand their symptoms and conditions in a casual, reassuring way. "        "You offer general advice and always recommend seeing a medical professional."    ),    verbose=True,    allow_delegation=False,    llm=llm,    memory=memory)predictive_diagnostic_agent = Agent(    role="Predictive Diagnostic Agent",    goal="Guide users by predicting diagnostic tests, treatment options, and expectations based on their provided medical condition.",    backstory=(        "You are a knowledgeable and supportive AI trained to interpret diagnosed medical conditions, "        "suggest necessary follow-up tests, provide possible treatments, and guide users through recovery steps. "        "While not a licensed doctor, you offer educational and compassionate responses."    ),    allow_delegation=False,    llm=llm)fitness_expert_agent = Agent(    role="AI Fitness Coach",    goal="Provide personalized fitness strategies, workouts, video resources, and AI-generated weight or health plans.",    backstory=(        "You're a virtual fitness coach specializing in body transformation, fat loss, muscle gain, posture, and endurance. "        "You understand health goals and suggest diet tips, YouTube videos (with thumbnails), equipment, and posture visuals. "        "Your goal is to educate, motivate, and guide users with effective personalized strategies."    ),    allow_delegation=False,    llm=llm)report_explainer = Agent(    role="Medical Report Explainer",    goal="Explain medical reports in plain, understandable language for non-experts",    backstory=(        "You're a friendly assistant who translates complex medical reports into simple explanations. "        "You help users understand their reports without giving diagnoses or medical advice."    ),    verbose=True,    allow_delegation=False,    llm=llm,    memory=memory)

In [4]:
def medipal_response(user_input):    task = Task(        description=(            f"The user asked: '{user_input}'. "            "Give a friendly, informative reply using previous context."        ),        expected_output=(            "A clear, compassionate, helpful response.\n"            "List down the important points in different lines.\n"            "Make it easy to understand for the user.\n"            "Always include a reminder to consult a doctor for medical advice."        ),        agent=medi_pal    )    crew = Crew(        agents=[medi_pal],        tasks=[task],        verbose=False    )    return crew.kickoff()

In [5]:
def diagnostic_response(condition):    description = (        f"The user has been diagnosed with: '{condition}'. "        "Please suggest:\n"        "1. Necessary diagnostic tests\n"        "2. Potential treatments (e.g., medications, therapies)\n"        "3. What to expect during treatment\n"        "4. Precautions or helpful advice\n\n"        "Reminder: Consult a licensed medical professional."    )    task = Task(        description=description,        expected_output=(            "1. List of diagnostic tests\n"            "2. Treatment options\n"            "3. Treatment journey details\n"            "4. Health tips or supportive advice\n"            "Reminder: Consult a licensed medical professional."        ),        agent=predictive_diagnostic_agent    )    crew = Crew(        agents=[predictive_diagnostic_agent],        tasks=[task],        verbose=False    )    return crew.kickoff()

In [6]:
def fitness_response(goal):    description = (        f"The user’s goal is: '{goal}'.\n"        "Using the Fitness Smart Search Tool, generate:\n"        "1. A weekly and daily structured workout plan (e.g., cardio, strength, flexibility)\n"        "2. List of recommended home/gym equipment\n"        "3. Curated YouTube videos (include title + URL)\n"        "4. Links to workout pose images or illustrations\n"        "5. A friendly health disclaimer and motivational note\n"        "\nMake it visually structured and actionable for the user."    )    expected_output = (        "Weekly Workout Plan (Day-wise)\n"        "Required Equipment\n"        "YouTube Video List (Title + URL)\n"        "Pose Image URLs\n"        "Health Disclaimer\n"        "Motivation to keep the user engaged\n"    )    task = Task(        description=description,        expected_output=expected_output,        agent=fitness_expert_agent    )    crew = Crew(        agents=[fitness_expert_agent],        tasks=[task],        verbose=False    )    return crew.kickoff()

In [7]:
def explain_report(file):    import fitz    doc = fitz.open(file.name)    full_text = "".join([page.get_text() for page in doc])    description = (        f"Explain the following medical report in plain language:\n\n{full_text}"    )    expected_output = (        "Break down the report into understandable sections.\n"        "Use simple explanations for medical terms.\n"        "Clarify test values (if any).\n"        "Avoid offering any medical diagnosis or advice.\n"        "Make the tone friendly and reassuring."    )    task = Task(        description=description,        expected_output=expected_output,        agent=report_explainer    )    crew = Crew(        agents=[report_explainer],        tasks=[task],        verbose=False    )    return crew.kickoff()

In [8]:
import gradio as grwith gr.Blocks() as demo:    gr.Markdown("## 🤖 MediPal - Your AI Health Assistant")    with gr.Tab("🩺 Symptom Checker"):        inp = gr.Textbox(label="Describe your symptoms")        out = gr.Textbox(label="MediPal Response", lines=8)        btn = gr.Button("Ask MediPal")        btn.click(fn=medipal_response, inputs=inp, outputs=out)    with gr.Tab("📋 Diagnostic Suggestions"):        cond = gr.Textbox(label="Enter diagnosed condition")        cond_out = gr.Textbox(label="Suggestions", lines=8)        cond_btn = gr.Button("Get Diagnostic Suggestions")        cond_btn.click(fn=diagnostic_response, inputs=cond, outputs=cond_out)    with gr.Tab("🏋️ Fitness Coach"):        goal = gr.Textbox(label="Enter fitness goal")        goal_out = gr.Textbox(label="Fitness Plan", lines=12)        goal_btn = gr.Button("Get Plan")        goal_btn.click(fn=fitness_response, inputs=goal, outputs=goal_out)    with gr.Tab("📄 Report Explainer"):        file = gr.File(label="Upload Medical Report (PDF)", file_types=[".pdf"])        file_out = gr.Textbox(label="Report Summary", lines=12)        file_btn = gr.Button("Explain Report")        file_btn.click(fn=explain_report, inputs=file, outputs=file_out)demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5dedb4a34d74b6a358.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
